# Pet segmentation: annotation audit

This notebook audits the Oxford-IIIT Pet trimaps for the **small annotation noise** and **other objects** found in the EDA,
and switches to an overlay that makes mask errors easier to see:

| Finding | What this notebook does |
|---|---|
| Thin strips of boundary pixels far from the pet (e.g. the chihuahua's left image edge) | Finds boundary (class 3) components that don't touch the pet interior, on every image |
| Stray foreground pixels inside the boundary band (e.g. the Bengal) | Finds pet (class 1) specks smaller than **0.1% of the image** |
| Connected-component counts need a minimum size, or noise reports fake "extra pets" | Counts animal components with and without the 0.1% minimum and lists the "fake extra pets" |
| The Persian lies on a person's arm: other objects touching the pet | Runs a COCO **Mask R-CNN** on the images, finds people and objects that touch the pet, and measures how their pixels are labeled |
| The green background tint makes edges hard to judge | New overlay: **only pet (red) and boundary (yellow) are colored**, with the animal outline drawn. Background stays as-is |

It also writes **cleaned trimaps**: detached boundary strips become background and tiny pet specks inside the band become
boundary. They're saved next to the originals and zipped to Drive. The raw files are not changed.

**Runtime:** *Runtime ▸ Change runtime type ▸ T4 GPU*, then *Run all*. The object scan is the slow part (about 10–15 min on a T4
for all images; set `OBJECT_SCAN_N` to scan a sample).

## 1. Setup

In [ ]:
import os, sys, random, shutil, tarfile, time, urllib.request, zipfile
from datetime import datetime
from pathlib import Path
from multiprocessing import Pool

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from scipy import ndimage
import torch
import torchvision
from tqdm.auto import tqdm

IN_COLAB = "google.colab" in sys.modules
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

MIN_FRAC = 0.001           # components smaller than 0.1% of the image are noise
EIGHT = np.ones((3, 3), dtype=bool)
OBJECT_SCAN_N = None       # None = every image; or e.g. 2000 for a random sample
DET_SCORE = 0.6            # minimum Mask R-CNN score for a detection
PET_ITSELF_FRAC = 0.5      # a detection whose pixels are mostly pet/boundary is the pet itself under a wrong class
REUSE_SCAN = True          # reuse objects_detected.csv from the latest runs/audit_* folder instead of re-running Mask R-CNN
SEED = 42
REQUIRE_GPU = True

DATA_DIR = Path("/content/dataset")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/upgaming/Animal Classification & Segmentation")
OUT_DIR = Path("/content/audit")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
elif REQUIRE_GPU:
    raise RuntimeError("No GPU found. In Colab choose Runtime ▸ Change runtime type ▸ T4 GPU, then run all again.")
else:
    device = torch.device("cpu")
    print("Running on CPU")
random.seed(SEED)
np.random.seed(SEED)

## 2. Data
Same loading as the training notebooks: `/content/dataset`, otherwise the archives in the Drive `Data` folder, otherwise
the official download. Every image listed in `trainval.txt` or `test.txt` is audited.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

IMAGES_DIR = DATA_DIR / "images"
ANN_DIR = DATA_DIR / "annotations"
TRIMAPS_DIR = ANN_DIR / "trimaps"
ARCHIVES = {
    "images.tar.gz": "https://thor.robots.ox.ac.uk/datasets/pets/images.tar.gz",
    "annotations.tar.gz": "https://thor.robots.ox.ac.uk/datasets/pets/annotations.tar.gz",
}


def dataset_ready():
    return IMAGES_DIR.is_dir() and TRIMAPS_DIR.is_dir() and (ANN_DIR / "trainval.txt").exists()


def extract(archive):
    print("extracting", archive.name)
    with tarfile.open(archive) as t:
        try:
            t.extractall(DATA_DIR, filter="data")
        except TypeError:  # Python without tarfile filters
            t.extractall(DATA_DIR)


if not dataset_ready():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    drive_data = DRIVE_PROJECT_DIR / "Data"
    for name, url in ARCHIVES.items():
        local = DATA_DIR / name
        on_drive = drive_data / name
        if on_drive.exists():
            print("copying", on_drive)
            shutil.copy(on_drive, local)
        else:
            print("downloading", url)
            urllib.request.urlretrieve(url, local)
        extract(local)
        local.unlink()
assert dataset_ready(), f"dataset not found in {DATA_DIR}"


def read_list(path):
    return [l.split()[0] for l in path.read_text().splitlines() if l.strip() and not l.startswith("#")]


def breed_of(name):
    return name.rsplit("_", 1)[0]


def species_of(name):
    return "cat" if name[0].isupper() else "dog"


split_of = {n: "test" for n in read_list(ANN_DIR / "test.txt")}
split_of.update({n: "trainval" for n in read_list(ANN_DIR / "trainval.txt")})
names = sorted(n for n in split_of if not n.startswith("._")
               and (IMAGES_DIR / f"{n}.jpg").exists() and (TRIMAPS_DIR / f"{n}.png").exists())
print(f"auditing {len(names)} images ({sum(split_of[n] == 'trainval' for n in names)} trainval, "
      f"{sum(split_of[n] == 'test' for n in names)} test)")

PET, BG, BND = 1, 2, 3


def load_trimap(name):
    return np.array(Image.open(TRIMAPS_DIR / f"{name}.png"))


def load_rgb(name):
    return np.array(Image.open(IMAGES_DIR / f"{name}.jpg").convert("RGB"))

## 3. Overlay that shows the mask edges
The EDA overlay tinted the background green, which hides where the mask ends. Here only the **pet interior (red)** and
**boundary band (yellow)** are tinted, the background keeps its real colours, and the outer edge of the animal
(pet + boundary) is drawn as a thin white line. Optional `highlight` pixels (noise found below) are drawn in magenta.

In [ ]:
PET_RGB, BND_RGB, EDGE_RGB, HI_RGB = np.array([228, 26, 28]), np.array([255, 217, 47]), np.array([255, 255, 255]), np.array([255, 0, 255])


def overlay(img, trimap, alpha=0.45, highlight=None, outline=True):
    out = img.astype(np.float32).copy()
    for v, col in [(PET, PET_RGB), (BND, BND_RGB)]:
        m = trimap == v
        out[m] = (1 - alpha) * out[m] + alpha * col
    if outline:
        animal = trimap != BG
        edge = animal & ~ndimage.binary_erosion(animal, EIGHT)
        out[edge] = EDGE_RGB
    if highlight is not None and highlight.any():
        ring = ndimage.binary_dilation(highlight, EIGHT, iterations=2)
        out[ring] = HI_RGB
    return out.clip(0, 255).astype(np.uint8)


def old_overlay(img, trimap):
    """The EDA's version: every class tinted, background green."""
    col = np.zeros((*trimap.shape, 3), np.uint8)
    col[trimap == PET], col[trimap == BG], col[trimap == BND] = [255, 0, 0], [0, 255, 0], [0, 0, 255]
    return (img * 0.6 + col * 0.4).astype(np.uint8)


demo =[next(n for n in names if breed_of(n) == b) for b in ["chihuahua", "Bengal", "Persian"] if any(breed_of(n) == b for n in names)] or names[:3]
fig, axes = plt.subplots(len(demo), 2, figsize=(11, 4.2 * len(demo)), squeeze=False)
for row, n in zip(axes, demo):
    img, t = load_rgb(n), load_trimap(n)
    row[0].imshow(old_overlay(img, t)); row[0].set_title(f"{n}: old overlay")
    row[1].imshow(overlay(img, t)); row[1].set_title(f"{n}: new overlay")
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Small annotation noise
For every trimap, with 8-connectivity and a minimum size of `MIN_FRAC` (0.1% of the image):

* **Detached boundary**: class-3 components that don't touch the pet interior (class 1) even after a 2-px dilation, like the strip
  on the chihuahua's image edge.
* **Pet specks**: class-1 components smaller than the minimum. A speck is **inside the band** when everything around it is
  boundary, like the Bengal's stray pixels.
* **Animal components** (pet + boundary): counted raw and with the minimum. Images where only the raw count is above 1 are
  the "fake extra pets" that a component count without a minimum would report.

In [ ]:
def audit_one(name):
    t = load_trimap(name)
    h, w = t.shape
    min_px = MIN_FRAC * t.size
    pet, bnd, animal = t == PET, t == BND, t != BG
    r = {"filename": name, "breed": breed_of(name), "species": species_of(name), "split": split_of[name],
         "height": h, "width": w}

    lab_a, _ = ndimage.label(animal, EIGHT)
    sizes_a = np.bincount(lab_a.ravel())[1:]
    r["animal_comps_raw"] = int(len(sizes_a))
    r["animal_comps_min"] = int((sizes_a >= min_px).sum())

    lab_p, _ = ndimage.label(pet, EIGHT)
    sizes_p = np.bincount(lab_p.ravel())[1:]
    speck_ids = np.nonzero(sizes_p < min_px)[0] + 1
    in_band = 0
    for i in speck_ids:
        comp = lab_p == i
        ring = ndimage.binary_dilation(comp, EIGHT) & ~comp
        if ring.any() and bnd[ring].all():
            in_band += 1
    r["pet_specks"] = int(len(speck_ids))
    r["pet_specks_in_band"] = int(in_band)
    r["pet_speck_px"] = int(sizes_p[sizes_p < min_px].sum())

    lab_b, nb = ndimage.label(bnd, EIGHT)
    near_pet = ndimage.binary_dilation(pet, EIGHT, iterations=2)
    touching = set(np.unique(lab_b[near_pet & bnd])) - {0}
    detached = [i for i in range(1, nb + 1) if i not in touching]
    det_mask = np.isin(lab_b, detached) if detached else np.zeros_like(bnd)
    r["detached_bnd_comps"] = len(detached)
    r["detached_bnd_px"] = int(det_mask.sum())
    r["detached_bnd_frac"] = det_mask.sum() / t.size
    border = np.zeros_like(bnd)
    border[0, :] = border[-1, :] = border[:, 0] = border[:, -1] = True
    r["detached_on_border"] = bool((det_mask & border).any())
    return r


if IN_COLAB:
    with Pool(os.cpu_count()) as pool:
        noise = pd.DataFrame(list(tqdm(pool.imap(audit_one, names, chunksize=32), total=len(names), desc="noise audit")))
else:
    noise = pd.DataFrame([audit_one(n) for n in tqdm(names, desc="noise audit")])

noise["fake_extra_pet"] = (noise.animal_comps_raw > 1) & (noise.animal_comps_min <= 1)
noise["any_noise"] = (noise.detached_bnd_comps > 0) | (noise.pet_specks > 0)
n = len(noise)
print(f"images with detached boundary strips: {(noise.detached_bnd_comps > 0).sum()} ({(noise.detached_bnd_comps > 0).mean():.1%}); "
      f"on the image border: {noise.detached_on_border.sum()}")
print(f"images with pet specks < {MIN_FRAC:.1%} of the image: {(noise.pet_specks > 0).sum()} ({(noise.pet_specks > 0).mean():.1%}); "
      f"specks inside the band: {noise.pet_specks_in_band.sum()} in {(noise.pet_specks_in_band > 0).sum()} images")
print(f"noise pixels: {noise.detached_bnd_px.sum() + noise.pet_speck_px.sum():,} "
      f"({(noise.detached_bnd_px.sum() + noise.pet_speck_px.sum()) / (noise.height * noise.width).sum():.4%} of all pixels)")
print(f"\nanimal components > 1: raw {(noise.animal_comps_raw > 1).sum()} images, "
      f"with the {MIN_FRAC:.1%} minimum {(noise.animal_comps_min > 1).sum()} images "
      f"-> {noise.fake_extra_pet.sum()} fake 'extra pets' removed by the minimum")
display(pd.DataFrame({"raw": noise.animal_comps_raw.clip(upper=5).value_counts().sort_index(),
                      f"min {MIN_FRAC:.1%}": noise.animal_comps_min.clip(upper=5).value_counts().sort_index()})
        .fillna(0).astype(int).rename_axis("animal components (5 = 5+)"))
display(noise.groupby("breed")[["any_noise", "fake_extra_pet"]].mean().sort_values("any_noise", ascending=False).head(10).round(3))

In [ ]:
def noise_mask(name):
    t = load_trimap(name)
    min_px = MIN_FRAC * t.size
    pet, bnd = t == PET, t == BND
    lab_p, _ = ndimage.label(pet, EIGHT)
    sizes_p = np.bincount(lab_p.ravel())
    specks = np.isin(lab_p, np.nonzero(sizes_p < min_px)[0]) & pet
    lab_b, nb = ndimage.label(bnd, EIGHT)
    touching = set(np.unique(lab_b[ndimage.binary_dilation(pet, EIGHT, iterations=2) & bnd])) - {0}
    detached = np.isin(lab_b, [i for i in range(1, nb + 1) if i not in touching])
    return t, specks, detached


def show_noise(sel, title, k=4):
    sel = sel[:k]
    if not sel:
        print(f"no examples for: {title}")
        return
    fig, axes = plt.subplots(1, len(sel), figsize=(4.6 * len(sel), 4.6), squeeze=False)
    for ax, n in zip(axes[0], sel):
        t, specks, detached = noise_mask(n)
        ax.imshow(overlay(load_rgb(n), t, highlight=specks | detached))
        ax.set_title(n, fontsize=10)
        ax.axis("off")
    fig.suptitle(title + " (magenta = noise)", fontsize=13)
    plt.tight_layout()
    plt.show()


show_noise(noise.sort_values("detached_bnd_px", ascending=False).query("detached_bnd_comps > 0").filename.tolist(),
           "Largest detached boundary strips")
show_noise(noise.query("pet_specks_in_band > 0").sort_values("pet_specks_in_band", ascending=False).filename.tolist(),
           "Pet specks inside the boundary band")
show_noise(noise.query("fake_extra_pet").filename.tolist(), "Fake 'extra pets' without a minimum size")
show_noise(noise.query("animal_comps_min > 1").filename.tolist(), "Still >1 animal component after the minimum (real 2nd pet or split mask)")

### Cleaned trimaps
Only the noise found above changes: **detached boundary → background**, **pet specks inside the band → boundary**. Everything
else is copied unchanged to `annotations/trimaps_clean/`. Training with `boundary_mode="ignore"` is barely affected
(the band is ignored anyway), but component counts and the "pet touches the border" statistics stop over-counting.

In [ ]:
CLEAN_TRIMAPS = ANN_DIR / "trimaps_clean"
CLEAN_TRIMAPS.mkdir(exist_ok=True)
changed = []
for n in tqdm(names, desc="writing cleaned trimaps"):
    t, specks, detached = noise_mask(n)
    lab_p, _ = ndimage.label(specks, EIGHT)
    fixed = t.copy()
    fixed[detached] = BG
    for i in range(1, lab_p.max() + 1):
        comp = lab_p == i
        ring = ndimage.binary_dilation(comp, EIGHT) & ~comp
        if ring.any() and (t[ring] == BND).all():
            fixed[comp] = BND
    diff = int((fixed != t).sum())
    if diff:
        changed.append({"filename": n, "changed_px": diff})
    Image.fromarray(fixed).save(CLEAN_TRIMAPS / f"{n}.png")
changed = pd.DataFrame(changed, columns=["filename", "changed_px"])
print(f"cleaned trimaps written to {CLEAN_TRIMAPS}: {len(changed)} of {len(names)} masks changed, "
      f"{changed.changed_px.sum():,} pixels in total")

## 5. Other objects touching the pet
A COCO-pretrained **Mask R-CNN** finds people and objects (everything except cats and dogs) in each image. For each object with
score ≥ `DET_SCORE` it records:

* whether it **touches the animal** (pet + boundary, after a 3-px dilation), like the arm under the Persian,
* how its pixels are **labeled in the trimap**: share background / pet / boundary.

If touching objects are mostly background, the annotators separated them cleanly. A noticeable pet/boundary share means
parts of people or objects were painted as the animal.

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights

weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
categories = weights.meta["categories"]
detector = maskrcnn_resnet50_fpn_v2(weights=weights).to(device).eval()
SKIP = {"__background__", "N/A", "cat", "dog"}

scan = names if OBJECT_SCAN_N is None else sorted(random.Random(SEED).sample(names, min(OBJECT_SCAN_N, len(names))))
previous = sorted((DRIVE_PROJECT_DIR / "runs").glob("audit_*/objects_detected.csv")) if REUSE_SCAN else []
if previous:
    print(f"reusing the saved scan {previous[-1]} (set REUSE_SCAN = False to re-run Mask R-CNN)")
    scan = names
else:
    print(f"scanning {len(scan)} images for other objects")

det_rows = []
BATCH = 8
with torch.no_grad():
    for b in tqdm(range(0, 0 if previous else len(scan), BATCH), desc="Mask R-CNN"):
        batch = scan[b:b + BATCH]
        imgs = [torch.from_numpy(load_rgb(n)).permute(2, 0, 1).float().div(255).to(device) for n in batch]
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            outs = detector(imgs)
        for n, o in zip(batch, outs):
            keep = o["scores"] >= DET_SCORE
            if not keep.any():
                continue
            t = load_trimap(n)
            animal_near = ndimage.binary_dilation(t != BG, EIGHT, iterations=3)
            for lbl, score, m, box in zip(o["labels"][keep].tolist(), o["scores"][keep].tolist(),
                                          o["masks"][keep][:, 0].float().cpu().numpy(), o["boxes"][keep].tolist()):
                cls = categories[lbl]
                if cls in SKIP:
                    continue
                obj = m > 0.5
                px = int(obj.sum())
                if px == 0:
                    continue
                det_rows.append({"filename": n, "breed": breed_of(n), "species": species_of(n), "object": cls,
                                 "score": round(score, 3), "object_px": px, "object_frac": px / t.size,
                                 "touches_pet": bool((obj & animal_near).any()),
                                 "labeled_bg": (t[obj] == BG).mean(), "labeled_pet": (t[obj] == PET).mean(),
                                 "labeled_boundary": (t[obj] == BND).mean()})

COLS = ["filename", "breed", "species", "object", "score", "object_px", "object_frac",
        "touches_pet", "labeled_bg", "labeled_pet", "labeled_boundary"]
dets = pd.read_csv(previous[-1])[COLS] if previous else pd.DataFrame(det_rows, columns=COLS)
dets["labeled_animal"] = dets.labeled_pet + dets.labeled_boundary
print(f"{len(dets)} objects in {dets.filename.nunique()} images; "
      f"{dets.touches_pet.sum()} touch the pet, in {dets[dets.touches_pet].filename.nunique()} images "
      f"({dets[dets.touches_pet].filename.nunique() / len(scan):.1%} of scanned images)")

### Drop detections that are the pet itself
The detector sometimes labels the cat or dog as another class: fluffy dogs become "bear" or "sheep", small pets become
"teddy bear". Those detections sit almost entirely on pet/boundary pixels, so they look like objects "painted as animal"
when they're really the animal. A detection is treated as **the pet itself** when at least `PET_ITSELF_FRAC` (50%) of its
pixels are labeled pet or boundary. Only the remaining detections count as other objects.

In [ ]:
dets["pet_itself"] = dets.labeled_animal >= PET_ITSELF_FRAC
raw_touch = dets[dets.touches_pet]
dropped = raw_touch[raw_touch.pet_itself]
print(f"dropped {len(dropped)} of {len(raw_touch)} touching detections as the pet itself "
      f"({len(dropped) / max(len(raw_touch), 1):.1%})")
display(dropped.object.value_counts().head(12).to_frame("dropped")
        .join(raw_touch.object.value_counts().rename("touching (raw)"))
        .assign(share_dropped=lambda d: (d.dropped / d["touching (raw)"]).round(3)))

In [ ]:
touch = dets[dets.touches_pet & ~dets.pet_itself]
if touch.empty:
    print("no objects touching a pet were found")
else:
    by_obj = (touch.groupby("object")
              .agg(objects=("filename", "size"), images=("filename", "nunique"),
                   labeled_bg=("labeled_bg", "mean"), labeled_pet=("labeled_pet", "mean"),
                   labeled_boundary=("labeled_boundary", "mean"),
                   absorbed_20pct=("labeled_animal", lambda s: (s > 0.2).mean()))
              .sort_values("objects", ascending=False))
    print("Objects touching the pet: mean share of their pixels per trimap label, and how often >20% of the object is painted as animal")
    display(by_obj.head(15).round(3))
    print(f"overall: touching objects are {touch.labeled_bg.mean():.1%} background, {touch.labeled_pet.mean():.1%} pet, "
          f"{touch.labeled_boundary.mean():.1%} boundary on average; "
          f"{(touch.labeled_animal > 0.2).mean():.1%} have >20% of their pixels labeled as animal")
    absorbed = lambda d: d.groupby("object").labeled_animal.apply(lambda s: (s > 0.2).mean())
    before_after = pd.DataFrame({"touching_raw": raw_touch.object.value_counts(), "touching_kept": touch.object.value_counts(),
                                 "absorbed_raw": absorbed(raw_touch), "absorbed_kept": absorbed(touch)}).fillna(0)
    print("\nBefore vs after dropping the pet-itself detections (absorbed = share of objects with >20% painted as animal):")
    display(before_after.sort_values("touching_raw", ascending=False).head(12).round(3))
    print(f"all objects: absorbed {(raw_touch.labeled_animal > 0.2).mean():.1%} -> {(touch.labeled_animal > 0.2).mean():.1%}")
    people = touch[touch.object == "person"]
    if len(people):
        print(f"people touching the pet: {len(people)} in {people.filename.nunique()} images "
              f"(top breeds: {', '.join(people.breed.value_counts().head(5).index)})")

In [ ]:
def show_objects(rows, title, k=6):
    rows = rows.head(k)
    if rows.empty:
        print(f"no examples for: {title}")
        return
    cols = 3
    nr = int(np.ceil(len(rows) / cols))
    fig, axes = plt.subplots(nr, cols, figsize=(5 * cols, 4.6 * nr), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), rows.iterrows()):
        img, t = load_rgb(r.filename), load_trimap(r.filename)
        with torch.no_grad():
            o = detector([torch.from_numpy(img).permute(2, 0, 1).float().div(255).to(device)])[0]
        sel = [i for i, (l, s) in enumerate(zip(o["labels"].tolist(), o["scores"].tolist()))
               if categories[l] == r.object and s >= DET_SCORE]
        obj = (o["masks"][sel][:, 0] > 0.5).any(0).cpu().numpy() if sel else np.zeros_like(t, bool)
        vis = overlay(img, t)
        edge = obj & ~ndimage.binary_erosion(obj, EIGHT, iterations=2)
        vis[edge] = [0, 255, 255]
        ax.imshow(vis)
        ax.set_title(f"{r.filename}: {r.object}\n{r.labeled_bg:.0%} bg / {r.labeled_pet:.0%} pet / {r.labeled_boundary:.0%} boundary",
                     fontsize=9)
    fig.suptitle(title + " (cyan = detected object outline)", fontsize=13)
    plt.tight_layout()
    plt.show()


show_objects(dropped.sort_values("object_frac", ascending=False), "Dropped: detections that are the pet itself")
show_objects(touch.query("object == 'person'").sort_values("object_frac", ascending=False), "People touching the pet")
show_objects(touch.query("object == 'person'").sort_values("labeled_animal", ascending=False), "People most painted as animal")
show_objects(touch.sort_values("labeled_animal", ascending=False), "Touching objects most painted as animal")
show_objects(touch.query("object != 'person'").sort_values("object_frac", ascending=False), "Other objects touching the pet")

## 6. Save the audit to Drive

In [ ]:
noise.to_csv(OUT_DIR / "noise_audit.csv", index=False)
changed.to_csv(OUT_DIR / "cleaned_trimaps_changes.csv", index=False)
dets.to_csv(OUT_DIR / "objects_detected.csv", index=False)
with zipfile.ZipFile(OUT_DIR / "trimaps_clean.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(CLEAN_TRIMAPS.glob("*.png")):
        z.write(p, f"trimaps_clean/{p.name}")
print("zip size:", f"{(OUT_DIR / 'trimaps_clean.zip').stat().st_size / 1e6:.1f} MB")

if IN_COLAB and DRIVE_PROJECT_DIR.exists():
    run_dir = DRIVE_PROJECT_DIR / "runs" / datetime.now().strftime("audit_%Y%m%d_%H%M%S")
    shutil.copytree(OUT_DIR, run_dir)
    print("saved to", run_dir)
else:
    print("results kept in", OUT_DIR)